In [1]:
!pip install ddgs trafilatura
!pip install openai-agents

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.1/67.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 827.0/827.0 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 15.0 MB/s eta 0:00:00


In [2]:
import os
import json
from pprint import pprint
from IPython.display import Markdown, display
from ddgs import DDGS
import trafilatura

from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from agents import Agent, Runner, function_tool

MODEL = "gpt-4.1-mini"

Step 1: Define the tools

In [3]:
@function_tool
def search_web(query:str):
  """Search the web using DuckDuckGo browser. Returns 3 results."""
  ddgs = DDGS()
  results = ddgs.text(query, max_results=3)
  print(f"  \u2705 search_web: Got results for {query}")
  return json.dumps(results, indent=2)

In [4]:
@function_tool
def fetch_url(url:str):
  """Fetch the content of a URL using trafilatura."""
  downloaded = trafilatura.fetch_url(url)
  if downloaded:
    text = trafilatura.extract(downloaded)
    if text:
      print(f"  \u2705 fetch_url: Got {len(text)} chars from {url[:60]}")
      return text
  print(f" \u274C fetch_url:Failed to get content from {url[:60]}")
  return f"Could not extract text from {url}. Try a different source."

Step 2: The agents

In [5]:
RESEARCH_AGENT_PROMPT = """
You are a research specialist. Your job is to research a given topic and produce
a comprehensive research brief.


You have access to two tools:
- search_web: Search the web for information
- fetch_url: fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results - which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 3 different sources, synthesize
into a research brief

You MUST gather information from at least 3 distinct sources before delivering
your brief.
If you have fewer than 3 sources, keep searching.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working - search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call - sometimes just thinking through what you
have is the right move.
"""
research_agent = Agent(
    name="Research Agent",
    instructions=RESEARCH_AGENT_PROMPT,
    model = MODEL,
    tools=[search_web,fetch_url]
)

In [6]:
WRITER_AGENT_PROMPT = """
You are a professional article writer.
You will receive a conversation history that includes two research briefs.
The orchestrator has already selected the best one — use ONLY the selected brief.
Ignore the rejected brief entirely.


Your job:
- Write a well-structured, engaging article based ONLY on the selected research brief
- Use a clear, conversational tone — write like a real blogger, not an academic
- Include relevant statistics and data points from the research
- Cite sources where appropriate using inline links
- Structure with a compelling headline, intro, body sections, and conclusion
- Aim for 800-1200 words


Do NOT ask for feedback, offer revisions, or include any commentary after the article.
Just deliver the finished article in markdown format.
"""
writer_agent=Agent(
    name="Writer Agent",
    instructions=WRITER_AGENT_PROMPT,
    model=MODEL
)

In [7]:
from agents import handoff

ORCHESTRATOR_AGENT_PROMPT = """
You are the orchestrator of a multi-agent article writing system.
Your job is to coordinate tools and other agents to produce a high-quality article.
Use the tools available to you and/or delegate tasks to the appropriate agents.
Never do the work yourself. Always use tools or agents.
Your tools and agents are specialists and should be doing the work, you are the manager.


Your process:
1. Use the research_agent tool twice (and ONLY twice) with slightly varying inputs to get 2 research briefs.
2. Pick the best research brief out of the two. Do not combine them, just pick the best one.
3. Hand off to the Writer Agent with the best research brief so it can write the article.


Do not do the research yourself or add anything, you MUST use the research_agent tool to get the briefs.
Do not write the article yourself, you MUST hand off to the Writer Agent.

"""
orchestrator_agent=Agent(
    name="Orchestrator Agent",
    instructions=ORCHESTRATOR_AGENT_PROMPT,
    model=MODEL,
    tools=[research_agent.as_tool(max_turns=30,tool_name="research_agent", tool_description="Research a topic and return a brief with key facts, statistics, themes and source URLs. Pass the topic as input.")],
    handoffs=[handoff(agent=writer_agent)]
)

Step 3: Input Guardrail

In [37]:
from agents import input_guardrail, GuardrailFunctionOutput

INPUT_GUARDRAIL_AGENT_PROMPT = """
Determine if the request involves BLOCKED topics:
- politics
- religion
- gambling
- weapons
Respond with ONLY "PASS" or "FAIL: <reason>". Nothing else.
"""
input_guardrail_agent = Agent(
    name="Input Guardrail Agent",
    instructions=INPUT_GUARDRAIL_AGENT_PROMPT,
    model= MODEL
)

@input_guardrail
async def topic_check(context, agent, input):
  result = await Runner.run(input_guardrail_agent, input = input)
  is_blocked = result.final_output.startswith("FAIL")
  print(f" {'\u274C' if is_blocked else '\u2705'} Topic check: {result.final_output}")
  return GuardrailFunctionOutput(output_info=result.final_output, tripwire_triggered=False) #is_blocked)

In [38]:
from agents import output_guardrail, GuardrailFunctionOutput

OUTPUT_GUARDRAIL_AGENT_PROMPT = """Determine if the final output result involves BLOCKED topics:
- politics
- religion
- gambling
- weapons
- psicological advice


Respond with ONLY "PASS" or "FAIL:<reason>". Nothing else.
"""

output_guardrail_agent = Agent(
    name="Output Guardrail Agent",
    instructions=OUTPUT_GUARDRAIL_AGENT_PROMPT,
    model=MODEL
)

@output_guardrail
async def article_check(context,agent,output):
  result = await Runner.run(output_guardrail_agent,input=output)
  is_blocked = result.final_output.startswith("FAIL")
  print(f" {'\u274C' if is_blocked else '\u2705'} Article check: {result.final_output}")
  return GuardrailFunctionOutput(output_info=result.final_output,tripwire_triggered=is_blocked)

In [39]:
# Update the Orchestrator Agent to include the input guardrail and the output guardrail
orchestrator_agent.input_guardrails = [topic_check]
orchestrator_agent.output_guardrails= [article_check]

In [40]:
result = await Runner.run(
    orchestrator_agent,
    input="Topic: Who will win the next US presidential election?",
    max_turns=30
)

 ❌ Topic check: FAIL: politics
  ✅ search_web: Got results for leading candidates for next US presidential election 2024 and their chances of winning
  ✅ search_web: Got results for current polls and expert predictions for the next US presidential election outcome
  ✅ fetch_url: Got 5854 chars from https://www.bbc.com/news/articles/cj4x71znwxdo
  ✅ fetch_url: Got 3137 chars from https://www.nytimes.com/interactive/2023/us/politics/preside
  ✅ search_web: Got results for 2024 US presidential election current polls expert predictions
  ✅ search_web: Got results for latest polling data for 2024 US presidential election Kamala Harris vs Donald Trump
  ✅ fetch_url: Got 1770 chars from https://www.racetothewh.com/president/polls
  ✅ fetch_url: Got 4788 chars from https://www.nytimes.com/interactive/2024/us/elections/polls-
  ✅ fetch_url: Got 4343 chars from https://www.reuters.com/graphics/USA-ELECTION/RESULTS/zjpqne
  ✅ fetch_url: Got 176055 chars from https://en.wikipedia.org/wiki/Nationwi

  ✅ fetch_url: Got 9833 chars from https://www.270towin.com/2024-election-forecast-predictions/


In [ ]:
display(Markdown(result.final_output))